# N11 Tape Seed Ablation

Runs seed ablations for the tape model family using the `2d_tape_ICNN.ipynb` configuration. The default focuses on the anisotropic structured Brazier ICNN model; uncomment the alternate architecture list to broaden the sweep.

In [ ]:
import os
from pathlib import Path

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from properties import TapeN11Properties
from run_architectures import SweepConfig

ROOT = Path.cwd()
train_file = "../experiment_data/tape_data/11_noded/n11_tape_train_dataset.npz"
valid_file = "../experiment_data/tape_data/11_noded/n11_tape_test_dataset.npz"
properties = TapeN11Properties(mass=-0.005)

# Copied from 2d_tape_ICNN.ipynb
K_init_chol = (0.2, 0.0, 0.1)
K_init_diag = (0.2, 0.1)

base_cfg = SweepConfig(
    der_K_diag=K_init_diag,
    der_K_chol=K_init_chol,
    hidden=(10,),
    corr_factor=0.01,
    input_mode="raw",
    only_stretching_NN=False,
    only_bending_NN=False,
    zero_reference=True,
    activation="tanh",
    mode="anisotropic",
    n_epochs=1000,
    lr=5e-2,
    weight_decay=1e-5,
    seed=42,
    valid_every=10,
    max_dlambda=1e-2,
    iters=10,
    ls_steps=10,
    abs_tol=5e-4,
    rel_tol=1e-4,
    early_stop=True,
    train_fail_on_nonconvergence=False,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=1e-6,
    hessian_reg_probes=1,
    hessian_reg_seed=0,
    force_key=None,
    force_loss_strength=0.0,
    force_components=(0, 1, 2),
    force_sign=1.0,
    return_loss_components=False,
    early_stopping=True,
    early_stopping_patience=200,
    early_stopping_min_delta=1e-5,
    early_stopping_warmup_epochs=200,
    restore_best_model=True,
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_force_predictions=False,
    plot_force_predictions=False,
    save_hessian_diagnostics=False,
    save_energy_landscapes=False,
    verbose=True,
    continue_on_failure=True,
    seed_list=tuple(range(26)),
)

print(properties)

OUTPUT_DIR = ROOT / "seed_ablation_outputs_n11_tape"
SUMMARY_DIR = OUTPUT_DIR / "seed_ablation_summary"
base_cfg = base_cfg.__class__(**{
    **base_cfg.__dict__,
    "output_dir": str(OUTPUT_DIR),
    "seed_list": base_cfg.seed_list,
})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving seed ablation results under: {OUTPUT_DIR.resolve()}")


TapeN11Properties(length=None, r0=0.005, axs=None, jxs=None, ixs1=None, ixs2=None, density=600.0, E=1000000.0, N=11, start=Array([0., 0., 0.], dtype=float64), end=Array([1.05660479, 0.        , 0.04239192], dtype=float64), mass=-0.005)
Saving seed ablation results under: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/seed_ablation_outputs_n11_tape


In [2]:
from run_architectures import subset_brazier_stiffness_only, subset_tape_tube_candidates

# Architectures already trained (results on disk; we only reload them, not retrain).
existing_architectures = [
    "diag_energy_mlp",
    "chol_energy_mlp",
    "diag_energy_icnn",
    "chol_energy_icnn",
    "diag_stiffness_mlp",
    "chol_stiffness_mlp",
    "brazier_chol_stiffness_mlp",
    "brazier_chol_stiffness_icnn",
]

# New architectures to train for the full 26-seed sweep.
new_architectures = [
    "brazier_diag_stiffness_mlp",
    "brazier_diag_stiffness_icnn",
]

# Combined list used by downstream plotting / diagnostics cells.
selected_architectures = existing_architectures + new_architectures

print(f"Total architectures for plotting: {len(selected_architectures)}")
print("  Existing (reload from disk):")
for name in existing_architectures:
    print(f"    - {name}")
print("  New (will be trained for all 26 seeds):")
for name in new_architectures:
    print(f"    - {name}")
print(f"Seeds: {base_cfg.seed_list}")


Total architectures for plotting: 10
  Existing (reload from disk):
    - diag_energy_mlp
    - chol_energy_mlp
    - diag_energy_icnn
    - chol_energy_icnn
    - diag_stiffness_mlp
    - chol_stiffness_mlp
    - brazier_chol_stiffness_mlp
    - brazier_chol_stiffness_icnn
  New (will be trained for all 26 seeds):
    - brazier_diag_stiffness_mlp
    - brazier_diag_stiffness_icnn
Seeds: (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25)


In [ ]:
from seed_ablation_utils import run_seed_ablation, load_seed_ablation_results

# Train only the new architectures for the full 26-seed sweep.
# The 8 existing architectures already have results.npz on disk (all 26/26 seeds);
# they are NOT retrained here and will be reloaded by cells 4 and 5.

print(f"Training {len(new_architectures)} new architectures across seeds {base_cfg.seed_list}:")
for name in new_architectures:
    print(f"  - {name}")

_ = run_seed_ablation(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    base_cfg=base_cfg,
    selected_architectures=new_architectures,
)

# Reload the full set (existing + new architectures, all 26 seeds each) from disk
# so cells 4 and 5 see the complete ablation.
all_seed_results = load_seed_ablation_results(
    str(OUTPUT_DIR),
    architectures=selected_architectures,
)
for arch_name in selected_architectures:
    n_loaded = len(all_seed_results.get(arch_name, []))
    print(f"  {arch_name}: {n_loaded} seeds loaded from disk")


Training 2 new architectures across seeds (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25):
  - brazier_diag_stiffness_mlp
  - brazier_diag_stiffness_icnn

SEED ABLATION for architecture: brazier_diag_stiffness_mlp

--- Running seed 0 for brazier_diag_stiffness_mlp ---

Running architecture: brazier_diag_stiffness_mlp
  model_cls               : StructuredBrazierDiagonalEnergyNN
  which_case              : MLP
  hidden                  : (10,)
  input_mode              : raw
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 0
  n_epochs                : 1000
  lr                      : 0.05
  weight_decay            : 1e-05
  max_dlambda             : 0.01
  iters                   : 10
  ls_steps                : 10
  abs_tol                 : 0.0005
  rel_tol                 : 0.0001
  early_stop

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/util.py:664: RuntimeWarning: Mass is non-positive; if this is because gravity is in +z, ignore, but if this is not the intention please correct.
  base, aux = get_slinky(properties)


Epoch 000 | Train total: 1.451e-01 | Train disp: 9.780e-03 | Train force: 0.000e+00 | Valid total: 1.157e-02 | Valid disp: 1.157e-02 | Valid force: 0.000e+00
Epoch 100 | Train total: 6.807e-02 | Train disp: 7.954e-03 | Train force: 0.000e+00 | Valid total: 8.899e-03 | Valid disp: 8.899e-03 | Valid force: 0.000e+00
Epoch 200 | Train total: 3.523e-02 | Train disp: 7.730e-03 | Train force: 0.000e+00 | Valid total: 8.925e-03 | Valid disp: 8.925e-03 | Valid force: 0.000e+00
Epoch 300 | Train total: 2.641e-02 | Train disp: 7.449e-03 | Train force: 0.000e+00 | Valid total: 8.669e-03 | Valid disp: 8.669e-03 | Valid force: 0.000e+00
Epoch 400 | Train total: 2.578e-02 | Train disp: 7.251e-03 | Train force: 0.000e+00 | Valid total: 8.524e-03 | Valid disp: 8.524e-03 | Valid force: 0.000e+00
Epoch 500 | Train total: 1.765e-02 | Train disp: 7.121e-03 | Train force: 0.000e+00 | Valid total: 8.457e-03 | Valid disp: 8.457e-03 | Valid force: 0.000e+00
Epoch 600 | Train total: 1.590e-02 | Train disp: 7.0

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/run_architectures.py:1038: RuntimeWarning: Mass is non-positive; if this is because gravity is in +z, ignore, but if this is not the intention please correct.
  base, aux = get_slinky(properties)
/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/architecture_plots.py:234: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



--- Running seed 1 for brazier_diag_stiffness_mlp ---

Running architecture: brazier_diag_stiffness_mlp
  model_cls               : StructuredBrazierDiagonalEnergyNN
  which_case              : MLP
  hidden                  : (10,)
  input_mode              : raw
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 1
  n_epochs                : 1000
  lr                      : 0.05
  weight_decay            : 1e-05
  max_dlambda             : 0.01
  iters                   : 10
  ls_steps                : 10
  abs_tol                 : 0.0005
  rel_tol                 : 0.0001
  early_stop              : True
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  early_stopping          : True
  early_stopping_patience : 200
  restore_best_model      : Tru

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/util.py:664: RuntimeWarning: Mass is non-positive; if this is because gravity is in +z, ignore, but if this is not the intention please correct.
  base, aux = get_slinky(properties)


Epoch 000 | Train total: 1.450e-01 | Train disp: 9.734e-03 | Train force: 0.000e+00 | Valid total: 1.150e-02 | Valid disp: 1.150e-02 | Valid force: 0.000e+00
Epoch 100 | Train total: 6.797e-02 | Train disp: 7.926e-03 | Train force: 0.000e+00 | Valid total: 8.836e-03 | Valid disp: 8.836e-03 | Valid force: 0.000e+00
Epoch 200 | Train total: 3.521e-02 | Train disp: 7.707e-03 | Train force: 0.000e+00 | Valid total: 8.869e-03 | Valid disp: 8.869e-03 | Valid force: 0.000e+00
Epoch 300 | Train total: 2.638e-02 | Train disp: 7.433e-03 | Train force: 0.000e+00 | Valid total: 8.623e-03 | Valid disp: 8.623e-03 | Valid force: 0.000e+00
Epoch 400 | Train total: 2.577e-02 | Train disp: 7.238e-03 | Train force: 0.000e+00 | Valid total: 8.486e-03 | Valid disp: 8.486e-03 | Valid force: 0.000e+00
Epoch 500 | Train total: 1.764e-02 | Train disp: 7.112e-03 | Train force: 0.000e+00 | Valid total: 8.425e-03 | Valid disp: 8.425e-03 | Valid force: 0.000e+00
Epoch 600 | Train total: 1.589e-02 | Train disp: 7.0

In [ ]:
import importlib
import seed_ablation_utils as sau

importlib.reload(sau)

# Load saved per-seed results; this does not retrain.
all_seed_results = sau.load_seed_ablation_results(OUTPUT_DIR)

sau.print_seed_summary(all_seed_results)
sau.save_seed_summary_json(all_seed_results, output_dir=str(SUMMARY_DIR))

sau.make_seed_ablation_plots(
    all_seed_results=all_seed_results,
    output_dir=str(SUMMARY_DIR),
    traj_idx=0,
)


##############################################################################################################
SEED ABLATION SUMMARY
##############################################################################################################

Architecture: brazier_chol_stiffness_icnn
  n_seeds           : 26
  n_success         : 26
  n_failed          : 0
  best_seed         : 13
  best final valid  : 5.652513e-03
  worst final valid : 5.928415e-03
  mean final valid  : 5.795079e-03
  std  final valid  : 6.106179e-05
  median final valid: 5.785724e-03
  mean final train  : 2.160938e-02
  std  final train  : 2.988104e-03

Architecture: brazier_chol_stiffness_mlp
  n_seeds           : 26
  n_success         : 26
  n_failed          : 0
  best_seed         : 8
  best final valid  : 9.756475e-03
  worst final valid : 9.940137e-03
  mean final valid  : 9.835059e-03
  std  final valid  : 4.867192e-05
  median final valid: 9.831202e-03
  mean final train  : 8.758271e-03
  std  final train

In [ ]:
from seed_ablation_utils import run_seed_hessian_diagnostics

run_seed_hessian_diagnostics(
    str(OUTPUT_DIR),
    use_predicted=True,
    splits=("train", "valid"),
    max_trajectories=1,
    stride=10,
    properties_class="TapeN11Properties",
)
print("Hessian diagnostics table:", OUTPUT_DIR / "hessian_diagnostics_table.csv")


/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/compute_architecture_hessian_diagnostics.py:524: RuntimeWarning: Mass is non-positive; if this is because gravity is in +z, ignore, but if this is not the intention please correct.
  base, aux = get_slinky(properties)


[ok] /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/seed_ablation_outputs_n11_tape/brazier_chol_stiffness_icnn__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_0__mdl_0.01__it_10__hreg_1e-06__hprobe_1__hseed_0__wd_1e-05__mode_anisotropic | train: M=7.817e+01, kappa=3.118e+04, states=2 | valid: M=4.620e+00, kappa=1.881e+03, states=2
[ok] /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/seed_ablation_outputs_n11_tape/brazier_chol_stiffness_icnn__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_10__mdl_0.01__it_10__hreg_1e-06__hprobe_1__hseed_0__wd_1e-05__mode_anisotropic | train: M=8.501e+01, kappa=3.391e+04, states=2 | valid: M=4.595e+00, kappa=1.869e+03, states=2
[ok] /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/seed_ablation_outputs_n11_tape/brazier_chol_stiffness_icnn__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_11__mdl_0.01__it_10__hreg_1e-06__hprobe_1__hseed_0__wd_1e-05__m

In [ ]:
print("Done. Key outputs:")
print("  - <architecture>__seed_*/results.npz")
print("  - seed_ablation_summary/*.json and *.png")
print("  - hessian_diagnostics_table.csv")


Done. Key outputs:
  - <architecture>__seed_*/results.npz
  - seed_ablation_summary/*.json and *.png
  - hessian_diagnostics_table.csv
